# 04 — Algorithmique

Recherche dichotomique, tris, méthode de Horner, arbres et graphes.

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.


## 4.1 Dichotomie (recherche binaire)

**Définition.** Dans une liste **triée**, on compare l'élément cherché au milieu et on élimine la moitié restante.

**Quand l'utiliser.** Uniquement si la liste est triée. Jamais sur une liste non triée (résultat faux).


In [ ]:
def dichotomie(x, L):
    g, d = 0, len(L) - 1
    while g <= d:                 # <= et pas < !
        m = (g + d) // 2
        if L[m] == x:
            return m
        elif L[m] < x:
            g = m + 1
        else:
            d = m - 1
    return None

L = [1, 3, 5, 7, 9, 11]
for i, v in enumerate(L):
    assert dichotomie(v, L) == i

assert dichotomie(4, L) is None
assert dichotomie(1, [1]) == 0
assert dichotomie(0, []) is None


**Pièges.** `while g < d` oublie le dernier élément ; `d = m` au lieu de `d = m - 1` peut boucler indéfiniment.

**Astuce.** Complexité O(log n) au pire, O(1) si x est exactement au milieu.


## 4.2 Tris par sélection et insertion

**Définition.** Deux tris itératifs en O(n²) : la sélection cherche le minimum restant ; l'insertion insère chaque élément dans la partie déjà triée.

**Quand l'utiliser.** Petites listes ; l'insertion est O(n) sur une liste presque triée. Éviter pour n > 10 000.


In [ ]:
def tri_selection(L):
    n = len(L)
    for i in range(n - 1):
        imin = i
        for j in range(i + 1, n):
            if L[j] < L[imin]:
                imin = j
        L[i], L[imin] = L[imin], L[i]

def tri_insertion(L):
    # meilleur cas O(n) si la liste est déjà triée
    for i in range(1, len(L)):
        x = L[i]
        j = i - 1
        while j >= 0 and L[j] > x:
            L[j + 1] = L[j]
            j -= 1
        L[j + 1] = x

import random
random.seed(42)
for tri in (tri_selection, tri_insertion):
    for _ in range(100):
        L = [random.randint(0, 50) for _ in range(random.randint(0, 20))]
        M = L.copy()
        tri(M)
        assert M == sorted(L)

# l'insertion est linéaire sur une liste déjà triée
deja = list(range(1000))
tri_insertion(deja)
assert deja == list(range(1000))


**Pièges.** Tri par insertion : pire cas O(n²) (liste inversée), meilleur cas O(n) (déjà triée).

**Astuce.** Vers n ≈ 10⁴, les tris O(n²) deviennent lents : préférer `sorted`.


## 4.3 Tri fusion (diviser pour régner)

**Définition.** On coupe la liste en deux, on trie chaque moitié récursivement, puis on fusionne en O(n).

**Quand l'utiliser.** Pour garantir O(n log n) dans tous les cas. Coût mémoire O(n) (pas en place).


In [ ]:
def fusion(A, B):
    res = []
    i = j = 0
    while i < len(A) and j < len(B):
        if A[i] <= B[j]:          # <= garantit la stabilité
            res.append(A[i])
            i += 1
        else:
            res.append(B[j])
            j += 1
    res.extend(A[i:])
    res.extend(B[j:])
    return res

def tri_fusion(L):
    if len(L) <= 1:
        return L
    m = len(L) // 2
    return fusion(tri_fusion(L[:m]), tri_fusion(L[m:]))

random.seed(7)
for _ in range(100):
    L = [random.randint(0, 50) for _ in range(random.randint(0, 30))]
    assert tri_fusion(L) == sorted(L)

# stabilité : les éléments égaux gardent leur ordre d'origine
paires = [(1, "a"), (0, "b"), (1, "c"), (0, "d")]
assert tri_fusion(paires) == [(0, "b"), (0, "d"), (1, "a"), (1, "c")]


**Pièges.** Le slicing `L[:m]` / `L[m:]` copie en O(n) ; oublier `<=` rend le tri instable.

**Astuce.** `<=` lors de la fusion conserve l'ordre des éléments égaux : tri **stable**.


## 4.4 Méthode de Horner

**Définition.** Évaluer P(x) = a₀ + a₁x + … + aₙxⁿ avec n multiplications et n additions, au lieu de O(n²).

**Quand l'utiliser.** Dès qu'un sujet demande d'évaluer un polynôme ; base des schémas de Hörner en concours.


In [ ]:
def horner(coeffs, x):
    # coeffs = [a0, a1, ..., an] du degré le plus bas au plus haut
    r = 0
    for a in reversed(coeffs):
        r = r * x + a
    return r

assert horner([1, 2, 3], 2) == 1 + 2 * 2 + 3 * 2**2   # 17
assert horner([], 5) == 0
assert horner([7], 5) == 7

# vérification sur un polynôme aléatoire
def evaluation_naive(coeffs, x):
    return sum(a * x**k for k, a in enumerate(coeffs))

random.seed(3)
for _ in range(50):
    deg = random.randint(0, 8)
    coeffs = [random.randint(-5, 5) for _ in range(deg + 1)]
    x = random.randint(-3, 3)
    assert horner(coeffs, x) == evaluation_naive(coeffs, x)


**Pièges.** Inverser l'ordre des coefficients ; oublier que Horner vaut aussi pour la division polynomiale.

**Astuce.** `reversed(coeffs)` parcourt du degré le plus haut au plus bas : `r = r*x + a`.


## 4.5 Comparer les complexités en pratique

**Définition.** Mesurer le temps d'exécution permet de *sentir* la différence entre O(n²), O(n log n) et O(n log n) natif.

**Quand l'utiliser.** Pour valider expérimentalement une complexité ; ne jamais conclure d'une seule mesure.


In [ ]:
import time

def chrono(f, L):
    M = L.copy()
    t0 = time.perf_counter()
    f(M)
    return time.perf_counter() - t0

random.seed(1)
L = [random.random() for _ in range(3000)]
print(f"insertion : {chrono(tri_insertion, L):.4f} s")
print(f"fusion    : {chrono(tri_fusion, L):.4f} s")
print(f"sorted    : {chrono(sorted, L):.5f} s")
assert sorted(L) == tri_fusion(L)


**Pièges.** Un chronométrage dépend de la machine et de la charge ; les `assert` de correction, pas de vitesse.

**Astuce.** Comparer les ordres de grandeur (×10, ×100) plutôt que les valeurs absolues.


## 4.6 Structures avancées : arbres binaires et graphes

**Définition.** Un arbre binaire a au plus deux enfants par nœud ; un graphe se représente par matrice d'adjacence ou listes de successeurs.

**Quand l'utiliser.** Arbres : parcours hiérarchiques. Graphes : réseaux, chemins, parcours BFS/DFS. *(Notions complémentaires, absentes des sources initiales.)*


In [ ]:
# --- Arbre binaire ---
class Noeud:
    def __init__(self, valeur, gauche=None, droite=None):
        self.valeur = valeur
        self.gauche = gauche
        self.droite = droite

def infixe(noeud):
    if noeud is None:
        return []
    return infixe(noeud.gauche) + [noeud.valeur] + infixe(noeud.droite)

def taille(noeud):
    return 0 if noeud is None else 1 + taille(noeud.gauche) + taille(noeud.droite)

def hauteur(noeud):
    if noeud is None:
        return 0
    return 1 + max(hauteur(noeud.gauche), hauteur(noeud.droite))

arbre = Noeud(2, Noeud(1), Noeud(3))
assert infixe(arbre) == [1, 2, 3]
assert taille(arbre) == 3
assert hauteur(arbre) == 2

# --- Graphe : matrice d'adjacence ---
n = 5
adj = [[0] * n for _ in range(n)]      # jamais [[0] * n] * n !

def ajoute_arete(adj, u, v):
    adj[u][v] = 1
    adj[v][u] = 1

ajoute_arete(adj, 0, 1)
ajoute_arete(adj, 1, 2)
ajoute_arete(adj, 3, 4)
assert adj[0][1] == 1 and adj[1][0] == 1
assert adj[0][2] == 0

# --- Graphe : listes de successeurs ---
succ = {i: [] for i in range(n)}

def ajoute(succ, u, v):
    succ[u].append(v)
    succ[v].append(u)

ajoute(succ, 0, 1)
ajoute(succ, 1, 2)
ajoute(succ, 3, 4)
assert succ[1] == [0, 2]

# --- BFS (file) : distances depuis 0 ---
from collections import deque

def bfs(succ, depart):
    dist = {depart: 0}
    file = deque([depart])
    while file:
        u = file.popleft()
        for v in succ[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                file.append(v)
    return dist

assert bfs(succ, 0) == {0: 0, 1: 1, 2: 2}

# --- DFS (pile) ---
def dfs(succ, depart):
    vus, ordre = set(), []
    pile = [depart]
    while pile:
        u = pile.pop()
        if u not in vus:
            vus.add(u)
            ordre.append(u)
            pile.extend(reversed(succ[u]))
    return ordre

assert dfs(succ, 0) == [0, 1, 2]


**Pièges.** Construire la matrice avec `[[0] * n] * n` (lignes partagées) ; oublier de marquer un sommet visité (boucle infinie).

**Astuce.** BFS = file (`deque.popleft`), DFS = pile (`list.pop`). La matrice d'adjacence teste une arête en O(1) mais coûte O(n²) en mémoire.


## Récapitulatif — réflexes concours

| Si le sujet demande… | J'utilise… |
|---|---|
| xⁿ ou puissance de matrice | exponentiation rapide O(log n) |
| chercher dans une liste triée | dichotomie O(log n) |
| trier avec garantie | tri fusion O(n log n) |
| évaluer un polynôme | Horner O(n) |
| historique / parenthésage | pile |
| plus court chemin en nombre d'arêtes | BFS (file) |
| justifier une complexité récursive | écrire T(n) puis résoudre |
